In [ ]:
!pip install optuna

In [ ]:
import pandas as pd
import torch
import optuna
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Load dataset
df = pd.read_csv("/content/drive/MyDrive/cleaned_dataset")
df.columns = df.columns.str.strip()

# Extract category labels
category_columns = df.columns[2:]  # Assuming first two columns are 'id' and 'message'
labels = df[category_columns].values  # Convert category columns to numpy array

# Tokenization
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

class DisasterDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(self.texts[idx], padding="max_length", truncation=True, max_length=128, return_tensors="pt")
        return {"input_ids": encoding["input_ids"].squeeze(),
                "attention_mask": encoding["attention_mask"].squeeze(),
                "labels": torch.tensor(self.labels[idx], dtype=torch.float)}

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(df["message"], labels, test_size=0.2, random_state=42)
train_dataset = DisasterDataset(X_train.tolist(), y_train)
test_dataset = DisasterDataset(X_test.tolist(), y_test)

# Compute metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = (torch.sigmoid(torch.tensor(logits)) > 0.5).int().numpy()  # Adjusted threshold
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="micro"),
        "precision": precision_score(labels, predictions, average="micro"),
        "recall": recall_score(labels, predictions, average="micro")
    }

# Hyperparameter tuning function
def model_pipeline(trial):
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 5e-5)
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32])
    epochs = trial.suggest_int("epochs", 3, 5)

    num_labels = labels.shape[1]  # Correct label count
    model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)

    training_args = TrainingArguments(
        output_dir="/content/drive/MyDrive/results",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        num_train_epochs=epochs,
        logging_dir="/content/drive/MyDrive/logs",
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()
    eval_results = trainer.evaluate()
    print(f"Epoch {epochs}: Accuracy: {eval_results['eval_accuracy']:.4f}, F1 Score: {eval_results['eval_f1']:.4f}")
    return eval_results["eval_f1"]

# Run hyperparameter tuning
study = optuna.create_study(direction="maximize")
study.optimize(model_pipeline, n_trials=3)

# Train final model with best parameters
best_params = study.best_params
print("Best Hyperparameters:", best_params)

num_labels = labels.shape[1]
final_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
final_trainer = Trainer(
    model=final_model,
    args=TrainingArguments
     (
        output_dir="/content/drive/MyDrive/final_model",
        per_device_train_batch_size=best_params["batch_size"],
        per_device_eval_batch_size=best_params["batch_size"],
        learning_rate=best_params["learning_rate"],
        num_train_epochs=best_params["epochs"],
        evaluation_strategy="epoch",
        logging_dir="/content/drive/MyDrive/logs",
        save_strategy="epoch",
        load_best_model_at_end=True
    ),
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

final_trainer.train()
final_trainer.save_model("/content/drive/MyDrive/best_disaster_model")
print("Best model saved.")

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"


In [ ]:
print(df.columns)  # Replace `df` with your actual DataFrame


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import shutil
shutil.move("/content/cleaned_dataset.csv", "/content/drive/MyDrive/cleaned_dataset")
